# Wat doet resamplen naar 16 kHz met de 384 kHz-opnames?

`src/preprocessdata.py` laadt elke opname met `SAMPLE_RATE = 16000`. Bij de 384 kHz-recorder gaat daardoor alles boven 8 kHz (de Nyquist-frequentie) verloren.

Vragen:
1. Hoeveel opnames en seconden komen van elke recorder, en welke labels?
2. Zit er in de 384 kHz-opnames boven 8 kHz energie die hommelgeluid (`buzz`, `flight`) onderscheidt van ruis, of `buzz` van `flight`?

Trello-kaart: *Uitzoeken wat resamplen naar 16 kHz doet met de 384 kHz-opnames* (lesweek 12). De conclusie staat ook in `docs/notitie-samplerate.md`.

In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import soundfile as sf

RAW_DATA_DIR = os.path.join("..", "data", "raw_data")
FEATURES_CSV = os.path.join("..", "data", "processed_data", "bee_features_per_second.csv")

BANDEN = [(0, 1_000), (1_000, 8_000), (8_000, 20_000), (20_000, 50_000)]
BAND_NAMEN = ["0-1 kHz", "1-8 kHz", "8-20 kHz", "20-50 kHz"]

## 1. Opnames per recorder

De sample rate lezen we uit de WAV-header met `soundfile.info`, zonder het geluid te laden.

In [ ]:
bestanden = sorted(n for n in os.listdir(RAW_DATA_DIR) if n.lower().endswith(".wav"))
info = pd.DataFrame(
    [
        {
            "source_file": n,
            "sample_rate": sf.info(os.path.join(RAW_DATA_DIR, n)).samplerate,
            "duur_s": sf.info(os.path.join(RAW_DATA_DIR, n)).duration,
            "dag": n[:8],
        }
        for n in bestanden
    ]
)
info.groupby("sample_rate").agg(
    opnames=("source_file", "count"),
    seconden=("duur_s", "sum"),
    dagen=("dag", lambda d: ", ".join(sorted(d.unique()))),
).round(0)

In [ ]:
features = pd.read_csv(FEATURES_CSV)
features["sample_rate"] = features["source_file"].map(info.set_index("source_file")["sample_rate"])
features["label"] = features["label"].fillna("(geen label)")

pd.crosstab(features["label"], features["sample_rate"], margins=True, margins_name="totaal")

**Wat we zien:**
- De 384 kHz-recorder levert maar 12 van de 150 opnames (ongeveer 12% van de seconden). Alle 16 kHz-opnames zijn van één dag (26-05-2026); de 384 kHz-opnames zijn van maart en april. **Recorder en opnamedag vallen dus samen.**
- De labels zijn ongelijk verdeeld over de recorders: `voice` komt bijna alleen voor bij 16 kHz, `radio` relatief vaak bij 384 kHz. Een model kan daardoor de recorder leren in plaats van het gedrag. Daarom moeten we de scores per recorder rapporteren (CLAUDE.md, doel 3).

## 2. Spectrogrammen van een buzz- en een flight-seconde (384 kHz)

De stippellijn ligt op 8 kHz: alles daarboven valt weg bij 16 kHz.

In [ ]:
def laad_seconde(source_file, start):
    """Eén seconde audio op de oorspronkelijke sample rate."""
    y, sr = sf.read(os.path.join(RAW_DATA_DIR, source_file), dtype="float32")
    begin_opname = pd.to_datetime(source_file[:15], format="%Y%m%d_%H%M%S")
    i = int((pd.Timestamp(start) - begin_opname).total_seconds())
    return y[i * sr : (i + 1) * sr], sr


hoog = features[features["sample_rate"] == 384_000]
voorbeelden = {
    label: hoog[hoog["label"] == label].iloc[len(hoog[hoog["label"] == label]) // 2]
    for label in ["buzz", "flight", "radio"]
}

fig, assen = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
for as_, (label, rij) in zip(assen, voorbeelden.items()):
    x, sr = laad_seconde(rij["source_file"], rij["start"])
    as_.specgram(x, NFFT=4096, Fs=sr, noverlap=2048, cmap="magma")
    as_.axhline(8_000, color="white", linestyle="--", linewidth=1)
    as_.set_ylim(0, 50_000)
    as_.set_title(f"{label}  ({rij['source_file']}, {rij['start'][11:]})")
    as_.set_xlabel("tijd (s)")
assen[0].set_ylabel("frequentie (Hz)")
plt.tight_layout()
plt.show()

## 3. Energie per frequentieband, per label (alle 384 kHz-seconden)

Per seconde berekenen we met een FFT de energie in vier banden, in dB. De mediaan per label laat zien welke banden verschillen.

In [ ]:
rijen = []
for source_file, groep in hoog.groupby("source_file"):
    y, sr = sf.read(os.path.join(RAW_DATA_DIR, source_file), dtype="float32")
    begin_opname = pd.to_datetime(source_file[:15], format="%Y%m%d_%H%M%S")
    for _, rij in groep.iterrows():
        i = int((pd.Timestamp(rij["start"]) - begin_opname).total_seconds())
        x = y[i * sr : (i + 1) * sr]
        if len(x) < sr:
            continue
        spectrum = np.abs(np.fft.rfft(x * np.hanning(len(x)))) ** 2
        freqs = np.fft.rfftfreq(len(x), 1 / sr)
        energie = [10 * np.log10(spectrum[(freqs >= a) & (freqs < b)].sum() + 1e-20) for a, b in BANDEN]
        rijen.append([source_file, rij["label"], *energie])

banden = pd.DataFrame(rijen, columns=["source_file", "label", *BAND_NAMEN])
banden.groupby("label")[BAND_NAMEN].median().round(1).assign(n=banden["label"].value_counts())

In [ ]:
volgorde = ["buzz", "flight", "noise", "radio", "voice", "(geen label)"]
fig, assen = plt.subplots(1, 2, figsize=(13, 4), sharey=True)
for as_, band in zip(assen, ["8-20 kHz", "20-50 kHz"]):
    data = [banden.loc[banden["label"] == l, band] for l in volgorde]
    as_.boxplot(data, tick_labels=volgorde, showfliers=False)
    as_.set_title(f"Energie {band} per label (384 kHz)")
    as_.set_ylabel("dB")
plt.tight_layout()
plt.show()

### Controle: geldt het verschil binnen dezelfde opname?

De 384 kHz-seconden komen uit maar 12 bestanden. We vergelijken daarom per opname de hommelseconden met de rest van dezelfde opname.

In [ ]:
per_opname = (
    banden.assign(groep=np.where(banden["label"].isin(["buzz", "flight"]), "hommel", "rest"))
    .groupby(["source_file", "groep"])["20-50 kHz"]
    .median()
    .unstack()
    .dropna()
)
per_opname["verschil_dB"] = per_opname["hommel"] - per_opname["rest"]
per_opname.round(1)

## Conclusie

- **Hommel tegenover ruis:** in de 384 kHz-opnames hebben `buzz` en `flight` boven 8 kHz duidelijk meer energie dan `radio` en `voice`, vooral tussen 20 en 50 kHz (mediaan 74 tot 76 dB tegenover 57 tot 61 dB). Binnen dezelfde opname geldt het ook: in 7 van de 8 opnames met hommelgeluid ligt de energie tussen 20 en 50 kHz 10 tot 17 dB hoger dan in de rest van de opname. In de achtste (`20260324_123001.WAV`) is er geen verschil; daar is de hommel waarschijnlijk zachter of verder weg. Resamplen naar 16 kHz gooit dus informatie weg die hommel van ruis onderscheidt.
- **Buzz tegenover flight:** boven 8 kHz lijken `buzz` en `flight` op elkaar. Voor dit onderscheid levert de hoge band weinig op; dat moet uit de band onder 8 kHz komen (grondfrequentie, harmonischen).
- **Maar:** de 16 kHz-recorder levert 88% van de data en kan boven 8 kHz niets meten. Features boven 8 kHz bestaan dus alleen voor 12 opnames. Een model dat ze gebruikt, wordt afhankelijk van de recorder. Bovendien zijn de aantallen klein (50 buzz- en 12 flight-seconden op 384 kHz) en vallen recorder en opnamedag samen.

Het advies staat in `docs/notitie-samplerate.md`.